# Scaling Study: All 5 Models (SP, optimal LR from sweep)

SVG Scaling Laws Project — Part 2

**Runtime**: A100 GPU

**Prerequisites**: Run `colab_lr_sweep.ipynb` first; its results must be on Drive.

**Estimated time**: ~2.5 hours (Tiny 5m + Small 10m + Medium 20m + Large 30m + XL 80m)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q /content/drive/MyDrive/SVGScalingLawProject/svg-scaling-project.zip -d /content/

In [ ]:
!pip install -q pyyaml mup

## 2. Verify

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

!ls /content/svg-scaling-project/data/tokenized/

## 3. Train All Models

In [ ]:
import subprocess, time, json, os

# --- Determine optimal LR from sweep results ---
# Per PDF spec Part 2: "Select the best learning rate based on validation
# loss. Use this same learning rate for all larger model sizes."
base_dir = '/content/svg-scaling-project'
sweep_dir = '/content/drive/MyDrive/SVGScalingLawProject/lr_sweep_colab_results'

best_loss = float('inf')
optimal_lr = None
for run_name in sorted(os.listdir(sweep_dir)):
    summary_path = os.path.join(sweep_dir, run_name, 'summary.json')
    if not os.path.exists(summary_path):
        continue
    with open(summary_path) as f:
        s = json.load(f)
    vl = s['best_val_loss']
    run_lr = s['config']['training']['learning_rate']
    print(f'  {run_name}: LR={run_lr:.0e}, best_val_loss={vl:.4f}')
    if vl < best_loss:
        best_loss = vl
        optimal_lr = run_lr

lr = f'{optimal_lr:.0e}'
print(f'\n→ Using optimal LR from sweep: {lr} (best_val_loss={best_loss:.4f})')

# --- Train all 5 models with the optimal LR ---
models = ['tiny', 'small', 'medium', 'large', 'xl']
results = []

for name in models:
    output_dir = f'{base_dir}/results/runs/scaling_study/{name}'
    print(f'\n{"=" * 60}')
    print(f'Training {name} (LR={lr})')
    print(f'{"=" * 60}')
    t0 = time.time()

    result = subprocess.run([
        'python', f'{base_dir}/src/train.py',
        '--config', f'{base_dir}/configs/{name}.yaml',
        '--learning-rate', lr,
        '--output-dir', output_dir,
        '--device', 'cuda',
    ], cwd=base_dir)

    elapsed = time.time() - t0
    status = 'OK' if result.returncode == 0 else f'FAIL (exit {result.returncode})'
    print(f'\n→ {name}: {status}, {elapsed/60:.1f} min')

    summary_path = f'{output_dir}/summary.json'
    try:
        with open(summary_path) as f:
            s = json.load(f)
        results.append(s)
        print(f'  best_val_loss={s["best_val_loss"]:.4f}, final_ppl={s["final_val_ppl"]:.2f}')
    except FileNotFoundError:
        print(f'  (no summary.json)')

print(f'\n{"=" * 60}')
print('All done!')
print(f'{"=" * 60}')
for r in results:
    print(f'  {r["config_name"]:>8s}: {r["n_params"]/1e6:.1f}M, '
          f'best_val_loss={r["best_val_loss"]:.4f}, ppl={r["final_val_ppl"]:.2f}')

## 4. Save Results to Drive

In [ ]:
import shutil, os

drive_dest = '/content/drive/MyDrive/SVGScalingLawProject/scaling_study_results'
os.makedirs(drive_dest, exist_ok=True)

for name in ['tiny', 'small', 'medium', 'large', 'xl']:
    src_dir = f'/content/svg-scaling-project/results/runs/scaling_study/{name}'
    dst_dir = f'{drive_dest}/{name}'
    os.makedirs(dst_dir, exist_ok=True)
    for fname in ['summary.json', 'training_log.json', 'best_model.pt']:
        src = f'{src_dir}/{fname}'
        if os.path.exists(src):
            shutil.copy2(src, f'{dst_dir}/{fname}')
            print(f'Copied {name}/{fname}')

print('\nAll results saved to Drive.')